<a id="inizio"></a>
# 11 · Capstone: il carico del Nord e la temperatura · Soluzioni

⏱ ~110 min &nbsp;·&nbsp; Dati: `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/load_total_north_hourly_2025.xlsx`, `../Dati/fallback/meteo_milano_2024_2025.json`

**In questo notebook impariamo a**

- caricare, unire e pulire una serie temporale reale, con i salti dell'ora legale
- costruire la serie giornaliera e i profili medi per ora e per giorno della settimana
- unire il carico con la temperatura di Open-Meteo e leggere la relazione tra i due

I dati utilizzati provengono dal [Download Center di Terna](https://dati.terna.it/en/download-center); in
particolare, useremo i dati di **carico elettrico della zona Nord** relativi agli anni 2024 e 2025. Nei
dataset Terna il carico è la potenza richiesta al sistema elettrico in ciascun intervallo temporale, una
grandezza centrale per la pianificazione e la gestione della rete di trasmissione.

Accanto al carico integriamo la **temperatura** di Milano, ottenuta tramite API da
[Open-Meteo](https://open-meteo.com). L'obiettivo è descrivere come il carico del Nord dipende dall'ora
del giorno, dal giorno della settimana e dalla temperatura. Il lavoro è diviso in sei step, ciascuno con
una consegna, una cella da completare e una verifica. Copilot si può usare per i titoli e le etichette
dei grafici, mentre il codice degli step lo scriviamo noi, perché lo scopo del capstone è mettere in
pratica quello che abbiamo visto nei notebook precedenti.

<a id="sez-1"></a>
## 1. Caricamento dei dati e unione

<a id="es-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 1 · Caricamento dei dati e unione**

In questo step costruiamo una singola serie storica continua a partire dai dati grezzi.

**Cosa fare.**

1. Leggi i due file Excel presenti nella cartella `../Dati/` in `df_2024` e `df_2025`:
   `load_total_north_hourly_2024.xlsx` e `load_total_north_hourly_2025.xlsx`.
2. Unisci i due dataset **per righe** in `df`, mantenendo lo stesso schema di colonne, in modo da
   ottenere una serie unica che copra l'intero periodo 2024-2025.
3. Controlla la dimensione del dataset, la prima e l'ultima data, il numero medio di righe per giorno
   e i valori mancanti per colonna. Il nome del file contiene `hourly`, quindi verifica con il numero
   di righe per giorno se la frequenza dei dati è effettivamente oraria.

*Suggerimento:* `df["Date"].dt.date.nunique()` conta i giorni diversi.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 1**

</div>

In [ ]:
import pandas as pd

df_2024 = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
df_2025 = pd.read_excel("../Dati/load_total_north_hourly_2025.xlsx")

df = pd.concat([df_2024, df_2025], ignore_index=True)

print(df.shape)
print(df["Date"].min(), df["Date"].max())
righe_giorno = len(df) / df["Date"].dt.date.nunique()
print(f"{righe_giorno:.0f} righe al giorno")
df.isna().sum()

In [ ]:
from corso import verifica
verifica("Step 1")

Il conto dà 96 righe al giorno, quindi i dati sono quartorari, con un valore ogni 15 minuti, anche se il
nome del file dice `hourly`. Ne terremo conto più avanti, quando dalla potenza in MW passeremo
all'energia in MWh, perché ogni valore copre un quarto d'ora.

<a id="sez-2"></a>
## 2. Colonna temporale e prima visualizzazione

<a id="es-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 2 · Preparazione della colonna temporale e prima visualizzazione**

In questo step rendiamo la colonna data utilizzabile come variabile temporale e facciamo una prima ispezione visiva della serie.

**Cosa fare.**

1. Controlla con `df.dtypes` che la colonna `Date` sia già di tipo `datetime`; se fosse letta come
   testo, la si converte con `pd.to_datetime`.
2. Ordina il dataset per data, perché ogni file comincia con l'ultimo quarto d'ora dell'anno, e
   reimposta l'indice per avere un ordinamento pulito.
3. Importa Plotly e visualizza **entrambe** le serie nel tempo: `Total Load [MW]` e
   `Forecast Total Load [MW]`.
4. Dopo aver verificato che le due curve siano coerenti, elimina le colonne che non useremo:
   `Forecast Total Load [MW]` e `Bidding Zone`.

Al termine di questo step deve rimanere una serie storica con una colonna temporale in formato
`datetime` e una sola colonna di valori, `Total Load [MW]`.

*Suggerimento:* `reset_index(drop=True)` rifà l'indice a partire da 0 senza conservare il vecchio come colonna, mentre `df.drop(columns=[...])` toglie le colonne indicate.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 2**

</div>

In [ ]:
import plotly.express as px

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

fig = px.line(df, x="Date", y=["Total Load [MW]", "Forecast Total Load [MW]"],
              title="Carico e previsione di Terna, zona Nord",
              labels={"value": "carico [MW]", "variable": "serie"})
fig.show()

df = df.drop(columns=["Forecast Total Load [MW]", "Bidding Zone"])
df.head()

In [ ]:
from corso import verifica
verifica("Step 2")

<a id="sez-3"></a>
## 3. Ora legale: duplicati e buchi

Prima di aggregare conviene controllare il passo tra un timestamp e il successivo, che in una serie
quartoraria regolare è sempre di 15 minuti. Il passo cambia soltanto nell'ultima domenica di marzo e
nell'ultima domenica di ottobre, i due giorni dell'anno in cui si passa dall'ora solare all'ora legale e
viceversa.

<a id="es-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 3 · Pulizia dei duplicati dell'ora legale**

In questo step lasciamo una riga per timestamp e contiamo i quarti d'ora che mancano.

**Cosa fare.**

1. Calcola in `passi` la differenza tra ogni timestamp e il precedente con `diff()`, conta con
   `value_counts()` quante volte compare ogni valore e chiediti quale sia il passo normale e a che
   cosa corrispondano gli altri due valori.
2. Metti in `duplicati` tutte le copie delle righe con un timestamp ripetuto, usando
   `duplicated(keep=False)`, e guarda in quali giorni e a quali ore cadono.
3. Ordina per `Date`, togli i duplicati con `drop_duplicates(subset="Date")` e reimposta l'indice.
4. Metti in `buchi` le righe che arrivano dopo un salto più lungo di 15 minuti e guarda in quali
   giorni cadono.

*Suggerimento:* l'ultima domenica di ottobre i quarti d'ora dalle 2:00 alle 2:45 compaiono due volte, mentre l'ultima domenica di marzo non esistono.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 3**

Dei due valori ripetuti teniamo il primo, perché le due copie differiscono di qualche centinaio di MW su circa 11.000 e la scelta incide poco sulle medie. Il buco di marzo invece resta, perché quei quarti d'ora non sono mai esistiti e non c'è nulla da ricostruire.

</div>

In [ ]:
passi = df["Date"].diff()
print(passi.value_counts())

duplicati = df[df["Date"].duplicated(keep=False)]
print(duplicati)

df = df.sort_values("Date").drop_duplicates(subset="Date").reset_index(drop=True)

buchi = df[df["Date"].diff() > pd.Timedelta("15min")]
buchi

In [ ]:
from corso import verifica
verifica("Step 3")

<a id="sez-4"></a>
## 4. Aggregazioni temporali e profili medi

Useremo `Date` come colonna temporale e `Total Load [MW]` come variabile di interesse.

<a id="es-4"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 4 · Aggregazioni temporali e profili medi**

In questo step analizziamo la serie quartoraria cambiando punto di vista sul tempo. Costruiamo una
serie giornaliera e osserviamo i pattern medi legati all'ora del giorno e al giorno della settimana.

**Cosa fare.**

**Serie giornaliera (somma)**

1. Parti dal DataFrame quartorario `df`.
2. Imposta la colonna `Date` come indice temporale in `df_indicizzato`.
3. Aggrega per giorno usando `resample("D")` e calcola la **somma** del carico.
4. Riporta `Date` come colonna e salva il risultato in `df_giornaliero`. Aggiungi poi la colonna
   `Energia [MWh]`, pari alla somma divisa per 4, perché un quarto d'ora a una potenza di P MW
   corrisponde a P/4 MWh.
5. Visualizza la serie giornaliera con Plotly.

**Profilo medio giornaliero (media per ora del giorno)**

6. Torna alla serie quartoraria `df`, estrai l'**ora del giorno** dalla colonna `Date` e salvala in
   una nuova colonna `hour`.
7. Raggruppa per `hour` e calcola la **media** del carico in `profilo_orario`.
8. Visualizza il profilo medio giornaliero con Plotly.

**Profilo medio settimanale (media per giorno della settimana)**

9. Sempre partendo da `df`, estrai il **giorno della settimana** dalla colonna `Date` e salvalo in
   una nuova colonna `weekday`, con la convenzione `0 = lunedì, …, 6 = domenica`.
10. Raggruppa per `weekday` e calcola la **media** del carico in `profilo_settimanale`.
11. Visualizza il profilo medio settimanale con Plotly.

Osserva infine i tre grafici e annota in quali periodi l'energia giornaliera scende, a che ore il
carico è più alto e di quanto cala nel weekend, perché questi numeri serviranno nello step 6.

*Suggerimento:* `dt.hour` e `dt.dayofweek` estraggono ora e giorno della settimana; `reset_index()` dopo `groupby` riporta la chiave come colonna.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 4**

L'energia giornaliera si ottiene sommando i valori quartorari e dividendo per 4, e non moltiplicando per 24 la potenza media del giorno. Nei due giorni di marzo con un'ora in meno, infatti, il prodotto per 24 conterebbe anche un'ora che non c'è stata.

</div>

In [ ]:
df_indicizzato = df.set_index("Date")
df_giornaliero = df_indicizzato["Total Load [MW]"].resample("D").sum()
df_giornaliero = df_giornaliero.reset_index()
df_giornaliero["Energia [MWh]"] = df_giornaliero["Total Load [MW]"] / 4
fig = px.line(df_giornaliero, x="Date", y="Energia [MWh]", title="Energia giornaliera, zona Nord")
fig.show()

df["hour"] = df["Date"].dt.hour
profilo_orario = df.groupby("hour")["Total Load [MW]"].mean().reset_index()
fig = px.line(profilo_orario, x="hour", y="Total Load [MW]", markers=True,
              title="Profilo medio giornaliero del carico",
              labels={"hour": "ora del giorno", "Total Load [MW]": "carico medio [MW]"})
fig.update_xaxes(dtick=1)
fig.show()

df["weekday"] = df["Date"].dt.dayofweek
profilo_settimanale = df.groupby("weekday")["Total Load [MW]"].mean().reset_index()
profilo_settimanale["giorno"] = ["lunedì", "martedì", "mercoledì", "giovedì", "venerdì", "sabato", "domenica"]
fig = px.line(profilo_settimanale, x="giorno", y="Total Load [MW]", markers=True,
              title="Profilo medio settimanale del carico",
              labels={"giorno": "giorno della settimana", "Total Load [MW]": "carico medio [MW]"})
fig.show()

In [ ]:
from corso import verifica
verifica("Step 4")

<a id="sez-5"></a>
## 5. Recupero e integrazione della temperatura

Al carico aggiungiamo la **temperatura a 2 metri** ricostruita per Milano. Si tratta di una
semplificazione, perché usiamo una sola città come riferimento per tutto il Nord Italia. I dati vengono
dall'API [Historical Weather](https://open-meteo.com/en/docs/historical-weather-api) di Open-Meteo, che
non richiede alcuna chiave e restituisce i valori orari per l'intervallo di date indicato.

La funzione `scarica_temperatura`, già scritta nelle celle qui sotto, chiede i dati all'API con
`requests.get` e restituisce la parte `hourly` della risposta, cioè un dizionario con le liste `time`
e `temperature_2m`. Se la rete non risponde, il blocco `try/except` legge al suo posto il file
`../Dati/fallback/meteo_milano_2024_2025.json`, che contiene temperature di esempio e non misurate.
In quel caso la funzione stampa un messaggio, e nelle frasi finali va indicato che la temperatura
viene dai dati di esempio.

In [ ]:
import json
from pathlib import Path

import requests

In [ ]:
def scarica_temperatura(start, end):
    """Temperatura oraria a Milano tra due date (Open-Meteo): dizionario con le liste time e temperature_2m."""
    url = "https://archive-api.open-meteo.com/v1/archive"
    parametri = {"latitude": 45.4642, "longitude": 9.19, "start_date": start, "end_date": end,
              "hourly": "temperature_2m", "timezone": "Europe/Rome"}
    try:
        response = requests.get(url, params=parametri, timeout=30)
        response.raise_for_status()
        return response.json()["hourly"]
    except requests.RequestException:
        file_meteo = Path("../Dati/fallback/meteo_milano_2024_2025.json")
        print(f"API non raggiungibile: uso i dati di esempio di {file_meteo}")
        with open(file_meteo, encoding="utf-8") as f:
            return json.load(f)["hourly"]

<a id="es-5"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 5 · Recupero e integrazione della temperatura**

In questo step integriamo il dato di temperatura con il dato di carico. Per prima cosa va risolta la
differente granularità, perché la temperatura è oraria mentre il carico è quartorario; la temperatura va
quindi portata da un valore ogni ora a un valore ogni 15 minuti.

**Cosa fare.**

1. Ricava dal carico `data_inizio` e `data_fine`, cioè il primo e l'ultimo giorno, scritti come testo
   nel formato `AAAA-MM-GG` con `.min()`, `.max()` e `.strftime("%Y-%m-%d")`.
2. Chiama `scarica_temperatura(data_inizio, data_fine)` e trasforma il risultato in `meteo` con
   `pd.DataFrame`.
3. Converti `time` in datetime, rinomina le colonne in `Date` e `Temperature_C`, imposta `Date` come
   indice e ordinalo con `sort_index()`.
4. Porta la temperatura a frequenza quartoraria con `resample("15min")` seguito da
   `interpolate(method="linear")`, così da poterla unire al carico, poi riporta `Date` come colonna e
   salva il risultato in `meteo_15`.
5. Fai il merge con il DataFrame del carico `df`, tenendo tutte le sue righe, e salva il risultato in
   `df_completo`.
6. Dopo il merge gli ultimi quarti d'ora del 2025 restano senza temperatura; riempili con `ffill()`,
   che ripete l'ultimo valore disponibile.
7. Visualizza la temperatura nel tempo con `px.line()`.

*Suggerimento:* `how="left"` tiene tutte le righe del DataFrame di sinistra e aggiunge la temperatura dove c'è. `resample("15min")` crea una riga ogni quarto d'ora, e `interpolate(method="linear")` riempie le righe nuove con valori intermedi.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 5**

L'interpolazione lineare stima i tre valori compresi tra un'ora e la successiva, quindi produce una stima e non una misura. Il merge con `how="left"` tiene tutte le righe del carico; le ultime tre del 2025, dopo le 23:00, restano senza temperatura perché la serie oraria finisce a quell'ora, e `ffill` le riempie con il valore delle 23:00.

</div>

In [ ]:
data_inizio = df["Date"].min().strftime("%Y-%m-%d")
data_fine = df["Date"].max().strftime("%Y-%m-%d")
print("Intervallo:", data_inizio, "->", data_fine)

orario = scarica_temperatura(data_inizio, data_fine)
meteo = pd.DataFrame(orario)
meteo["time"] = pd.to_datetime(meteo["time"])
meteo = meteo.rename(columns={"time": "Date", "temperature_2m": "Temperature_C"})
meteo = meteo.set_index("Date").sort_index()

meteo_15 = meteo.resample("15min").interpolate(method="linear")
meteo_15 = meteo_15.reset_index()

df_completo = pd.merge(df, meteo_15, on="Date", how="left")
print(df_completo["Temperature_C"].isna().sum(), "temperature mancanti dopo il merge")
df_completo["Temperature_C"] = df_completo["Temperature_C"].ffill()

fig = px.line(df_completo, x="Date", y="Temperature_C", title="Temperatura a Milano",
              labels={"Temperature_C": "temperatura [°C]"})
fig.show()

In [ ]:
from corso import verifica
verifica("Step 5")

<a id="sez-6"></a>
## 6. Carico e temperatura

<a id="es-6"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 6 · Carico e temperatura**

In questo step costruiamo uno scatter con un punto per giorno, la temperatura media sulle x, il carico medio sulle y e un colore diverso per ogni mese. Dalla forma della nuvola di punti si legge come il carico dipende dalla temperatura.

**Cosa fare.**

1. Calcola in `giornaliero` le medie giornaliere di `df_completo`, impostando `Date` come indice,
   selezionando le colonne `Total Load [MW]` e `Temperature_C`, applicando `resample("D").mean()` e
   riportando infine `Date` come colonna con `reset_index()`.
2. Aggiungi la colonna `mese` con il nome del mese, ricavato con `dt.month_name()`, in modo che lo
   scatter abbia un colore per ogni mese invece di una scala continua.
3. Disegna lo scatter con `px.scatter`, con la temperatura sulle x, il carico sulle y e
   `color="mese"`, e osserva che forma ha la nuvola e a quale temperatura il carico è più basso.
4. Completa le cinque frasi della cella che segue la verifica, usando i numeri e i grafici ottenuti
   negli step precedenti.

*Suggerimento:* con `dt.month`, che restituisce un numero, Plotly usa una scala di colore continua, mentre con `dt.month_name()` ogni mese ha un colore distinto.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 6**

Le medie giornaliere eliminano il ciclo delle ore all'interno della giornata e lasciano un solo valore per giorno, che si può confrontare direttamente con la temperatura media dello stesso giorno.

</div>

In [ ]:
df_indicizzato = df_completo.set_index("Date")
giornaliero = df_indicizzato[["Total Load [MW]", "Temperature_C"]].resample("D").mean()
giornaliero = giornaliero.reset_index()
giornaliero["mese"] = giornaliero["Date"].dt.month_name()

fig = px.scatter(giornaliero, x="Temperature_C", y="Total Load [MW]", color="mese", hover_data=["Date"],
                 title="Carico medio giornaliero e temperatura media, zona Nord 2024-2025",
                 labels={"Temperature_C": "temperatura media [°C]", "Total Load [MW]": "carico medio [MW]"})
fig.show()

In [ ]:
from corso import verifica
verifica("Step 6")

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Le cinque frasi**

1. I dati sono quartorari, non orari: 70.168 righe pulite, 8 doppioni di ottobre tolti (4 per anno),
   8 quarti d'ora di marzo mancanti, che restano un buco.
2. L'energia giornaliera va da circa 280.000 MWh (1° gennaio 2024) a circa 645.000 MWh (luglio); i
   giorni più leggeri sono le feste, le domeniche e agosto.
3. Nella giornata media il carico è minimo alle 3 (circa 14.300 MW) e massimo tra le 9 e le 11
   (circa 23.500 MW), con un secondo rialzo verso le 19.
4. La domenica il carico medio sta intorno a 15.300 MW, un giorno feriale tra 20.400 e 21.900 MW.
5. Il carico è più basso nelle giornate miti, intorno ai 15-20 °C, e sale con il freddo e con il caldo;
   al Nord pesa di più il caldo, con i massimi a luglio. I valori esatti dipendono dalla fonte della
   temperatura.

</div>

---

**Fine del notebook 11.** Precedente: [10 · Agenti per il coding](10_Agenti_per_il_coding.ipynb)

<sub>Blocco 4 · Agenti per il coding e caso d'uso end-to-end · Giornata 2 · Aula Base</sub>